# Gemma 4 E4B LoRA 파인튜닝 (Unsloth)

데비&마를렌 캐릭터 말투 학습 (8B, QLoRA 4-bit)

## 사용법
1. `train_final.json`을 Google Drive 루트에 업로드
2. 런타임 > 모두 실행
3. 셀 1에서 설치 후 자동 재시작됨 → 다시 런타임 > 모두 실행

In [ ]:
#@title 1. Unsloth 설치 (재시작 1회 필요)
try:
    import unsloth
    print(f"Unsloth {unsloth.__version__} 이미 설치됨")
except ImportError:
    print("Unsloth 설치 중...")
    !pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
    print("설치 완료 - 런타임 재시작합니다. 다시 '런타임 > 모두 실행' 하세요.")
    import os
    os.kill(os.getpid(), 9)

In [ ]:
#@title 2. 설정 + Drive 마운트
import os, shutil, torch

CHARACTER = "debi-marlene"
MODEL_NAME = "unsloth/gemma-4-E4B-it"

NUM_EPOCHS = 10
LEARNING_RATE = 5e-5
LORA_RANK = 16
LORA_ALPHA = 32
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_LENGTH = 512

OUTPUT_DIR = f"/content/output/{CHARACTER}"
DRIVE_BACKUP = f"/content/drive/MyDrive/gemma4_backup/{CHARACTER}"

# 이전 학습 결과 정리
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_BACKUP, exist_ok=True)

if os.path.exists("/content/sample_data"):
    shutil.rmtree("/content/sample_data")

print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"배치: {BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}, LR: {LEARNING_RATE}, 에폭: {NUM_EPOCHS}")

In [ ]:
#@title 3. 모델 로딩 + LoRA (Unsloth)
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"LoRA: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")

In [ ]:
#@title 4. 데이터셋 준비
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

search_paths = [
    "/content/drive/MyDrive/train_final.json",
    f"/content/drive/MyDrive/{CHARACTER}/train_final.json",
    f"{DRIVE_BACKUP}/train_final.json",
    "/content/drive/MyDrive/qwen35_backup/debi-marlene/train_final.json",
    "/content/drive/MyDrive/qwen3_omni_backup/debi-marlene/train_final.json",
]
data_path = next((p for p in search_paths if os.path.exists(p)), None)
if not data_path:
    raise FileNotFoundError("train_final.json을 Google Drive 루트에 업로드하세요")

with open(data_path) as f:
    raw_data = json.load(f)
print(f"데이터: {len(raw_data)}개")

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def format_to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_to_text, remove_columns=dataset.column_names)
print(f"샘플:\n{dataset[0]['text'][:300]}")

steps_per_epoch = max(1, len(raw_data) // (BATCH_SIZE * GRAD_ACCUM))
print(f"\nsteps/epoch: ~{steps_per_epoch}, 총: ~{steps_per_epoch * NUM_EPOCHS}")

In [ ]:
#@title 5. 학습
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_LENGTH,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=7,
        bf16=True,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=2,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
)

print(f"학습 시작 ({NUM_EPOCHS} 에폭)")
stats = trainer.train()
print(f"\n학습 완료: loss={stats.metrics['train_loss']:.4f}")

In [ ]:
#@title 7. loss 확인
print("=== Loss 곡선 ===")
for entry in trainer.state.log_history:
    if "loss" in entry:
        print(f"  step {entry['step']:>3}: loss={entry['loss']:.4f} (epoch {entry.get('epoch', '?'):.1f})")

In [ ]:
#@title 7. 추론 테스트

SYSTEM = ("너는 이터널 리턴의 쌍둥이 실험체 데비&마를렌이야. "
          "데비(언니)는 활발하고 천진난만하며 장난스러운 말투를 써. "
          "마를렌(동생)은 냉소적이고 신중하며 간결한 말투를 써.")

for q in ["안녕!", "너 누구야?", "10킬 했어!", "계속 져...", "데비야 결혼해줘", "심심해"]:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, temperature=0.5, top_p=0.9, do_sample=True)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {resp}\n")

In [ ]:
#@title 8. Drive 백업
import shutil, glob

save_path = f"{DRIVE_BACKUP}/adapter_lora"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

if data_path:
    shutil.copy2(data_path, DRIVE_BACKUP)

print(f"백업: {save_path}")
for f in sorted(glob.glob(f"{save_path}/*")):
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1024/1024:.1f} MB)")